# 03 — Build Site

Queries `data/elo.db`, builds leaderboard and player histories, generates `index.html`.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta

sys.path.insert(0, str(Path('..').resolve()))
DB_PATH = str(Path('../data/elo.db'))
SITE_PATH = Path('../index.html')

In [ ]:
from src.db import create_db, get_player_history
conn = create_db(DB_PATH)
print('DB opened.')

In [ ]:
cutoff = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')

rows = conn.execute("""
    SELECT p.player_id, p.name, pr.rating_after AS current_rating,
           COUNT(DISTINCT pr_all.match_id) AS matches_played
    FROM players p
    JOIN player_ratings pr ON p.player_id = pr.player_id
    JOIN (SELECT player_id, MAX(date) AS d FROM player_ratings GROUP BY player_id) l
      ON pr.player_id = l.player_id AND pr.date = l.d
    JOIN player_ratings pr_all ON p.player_id = pr_all.player_id
    GROUP BY p.player_id
    ORDER BY pr.rating_after DESC
""").fetchall()

players = []
for row in rows:
    pid, name, cur, matches = row[0], row[1], row[2], row[3]

    old = conn.execute("""
        SELECT rating_after FROM player_ratings
        WHERE player_id = ? AND date <= ?
        ORDER BY date DESC LIMIT 1
    """, (pid, cutoff)).fetchone()
    delta_30d = cur - (old[0] if old else cur)

    recent = conn.execute("""
        SELECT rating_after FROM player_ratings
        WHERE player_id = ? ORDER BY date DESC LIMIT 20
    """, (pid,)).fetchall()
    recent_ratings = [r[0] for r in reversed(recent)]

    players.append({
        'player_id': pid, 'name': name, 'current_rating': cur,
        'rating_delta_30d': delta_30d, 'matches_played': matches,
        'recent_ratings': recent_ratings
    })

print(f'Loaded {len(players)} players. Top 5:')
for p in players[:5]:
    print(f'  {p["name"]}: {p["current_rating"]:.1f} ({p["rating_delta_30d"]:+.1f} 30d)')

In [ ]:
player_histories = {p['player_id']: get_player_history(conn, p['player_id']) for p in players}
print(f'Built histories for {len(player_histories)} players.')

In [ ]:
from src.site_builder import build_site

html = build_site(players, player_histories)
SITE_PATH.write_text(html, encoding='utf-8')
size_mb = SITE_PATH.stat().st_size / 1_000_000
print(f'Written {size_mb:.1f} MB to {SITE_PATH.resolve()}')
print(f'Open in browser: file:///{SITE_PATH.resolve()}')